In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

import os
import glob
import warnings
import wandb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    f1_score,
    recall_score,
    confusion_matrix,
    mean_absolute_error
)

from scipy.stats import pearsonr


# Wandb path to artifact and local path to saved params

In [ ]:
paths={}

# Model

In [ ]:
class MultiHeadNet(nn.Module):
    def __init__(self,
                 input_dim: int,
                 trunk_layers: int,
                 trunk_dim: int,
                 reg_layers: int,
                 reg_hidden: int,
                 cls_layers: int,
                 cls_hidden: int,
                 dropout_rate: float):
        super().__init__()
        # ── Shared trunk ─────────────────────────────────────────────
        trunk = []
        prev_dim = input_dim
        for _ in range(trunk_layers):
            trunk += [
                nn.Linear(prev_dim, trunk_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_rate),
            ]
            prev_dim = trunk_dim
        self.trunk = nn.Sequential(*trunk)

        # ── Regression head ─────────────────────────────────────────
        reg = []
        prev = trunk_dim
        for _ in range(reg_layers):
            reg += [
                nn.Linear(prev, reg_hidden),
                nn.ReLU(inplace=True),
                #nn.Dropout(dropout_rate),
            ]
            prev = reg_hidden
        reg += [nn.Linear(prev, 1)]
        self.reg_head = nn.Sequential(*reg)

        # ── Classification head ────────────────────────────────────
        cls = []
        prev = trunk_dim
        for _ in range(cls_layers):
            cls += [
                nn.Linear(prev, cls_hidden),
                nn.ReLU(inplace=True),
                #nn.Dropout(dropout_rate),
            ]
            prev = cls_hidden
        cls += [nn.Linear(prev, 1)]
        self.cls_head = nn.Sequential(*cls)

    def forward(self, x: torch.Tensor):
        """
        x: Tensor of shape (N, input_dim)
        Returns:
            reg_out: Tensor of shape (N,)       # regression output
            cls_logits: Tensor of shape (N,)    # pre-sigmoid classification logit
        """
        features = self.trunk(x)
        reg_out = self.reg_head(features)
        cls_logits = self.cls_head(features)
        return reg_out.squeeze(-1), cls_logits.squeeze(-1)

# Functions

In [ ]:
def _safe_avg_precision(y_true, y_score):
    """Average Precision needs at least one positive; else NaN."""
    y_true = np.asarray(y_true)
    if np.sum(y_true == 1) == 0:
        return np.nan
    return average_precision_score(y_true, np.asarray(y_score))

def _safe_auc(y_true, y_score):
    """ROC AUC needs both classes present; else NaN."""
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return np.nan
    return roc_auc_score(y_true, np.asarray(y_score))

def _safe_f1_at_threshold(y_true, y_score, thr=0.5, pos_label=1):
    """Threshold scores → hard labels, guard single-class slices."""
    y_true = np.asarray(y_true)
    if np.unique(y_true).size < 2:
        return np.nan
    y_pred = (np.asarray(y_score) >= thr).astype(int)
    return f1_score(y_true, y_pred, pos_label=pos_label, zero_division=0)

def _safe_recall_at_threshold(y_true, y_score, thr=0.5, pos_label=1):
    """Recall with thresholding; NaN if no positives in y_true."""
    y_true = np.asarray(y_true)
    if np.sum(y_true == pos_label) == 0:
        return np.nan
    y_pred = (np.asarray(y_score) >= thr).astype(int)
    return recall_score(y_true, y_pred, pos_label=pos_label, zero_division=0)

# Map of metric names → callables that accept (y_true, y_score)
METRIC_FUNCS = {
    "avg_precision": _safe_avg_precision,
    "auc": _safe_auc,

    "mae": mean_absolute_error,
    "pearson": lambda y_true, y_score: pearsonr(y_true, y_score)[0],
    "confusion_matrix": lambda y_true, y_score: confusion_matrix(y_true, (np.asarray(y_score) >= 0.5).astype(int))
}


def compute_cellwise_metric_heatmap(
    df: pd.DataFrame,
    model: nn.Module,
    cols_rem: list[str],
    metric_name: str,
    *,
    group_cols=("Concentration", "Timepoint"),
    target_col="is_Active",
    device: torch.device | str = "cpu",
    cls_threshold: float = 0.5,
    logits_are_probs: bool = False,
):
    """
    Returns a DataFrame with metric values per (Concentration, Timepoint) cell.
    Assumes model(X)[1] are classification logits or probabilities.

    Parameters
    ----------
    df : DataFrame with group_cols and target_col.
    model : torch.nn.Module
        Forward should return (reg_out, cls_logits) or similar; we use [1].
    cols_rem : list[str]
        Columns to drop to form X features.
    metric_name : {"avg_precision","auc","f1","recall"}
    group_cols : tuple[str, str]
        ('Concentration','Timepoint') by default.
    target_col : str
        Classification target column (0/1).
    device : torch.device | str
        Device for inference.
    cls_threshold : float
        Threshold for f1/recall.
    logits_are_probs : bool
        Set True if model already outputs probabilities in head [1].
    """
    model.eval()
    device = torch.device(device)
    model.to(device)

    # Preallocate grid
    c_vals = np.sort(df[group_cols[0]].unique())
    t_vals = np.sort(df[group_cols[1]].unique())
    stats = pd.DataFrame(index=c_vals, columns=t_vals, dtype=float)

    # Choose the metric function
    if metric_name in ("avg_precision", "auc"):
        metric_fn = METRIC_FUNCS[metric_name]
    elif metric_name == "f1":
        metric_fn = lambda yt, ys: _safe_f1_at_threshold(yt, ys, thr=cls_threshold)
    elif metric_name == "recall":
        metric_fn = lambda yt, ys: _safe_recall_at_threshold(yt, ys, thr=cls_threshold)
    else:
        raise ValueError(f"Unknown metric_name: {metric_name}")

    # Loop over cells
    for c in c_vals:
        for t in t_vals:
            mask = (df[group_cols[0]] == c) & (df[group_cols[1]] == t)
            part = df.loc[mask]
            if part.empty:
                stats.loc[c, t] = np.nan
                continue

            y_true = part[target_col].to_numpy()

            # Build features and run model
            X_np = part.drop(columns=cols_rem).to_numpy(dtype=np.float32, copy=False)
            X_t = torch.from_numpy(X_np).to(device)

            with torch.no_grad():
                out = model(X_t)
                scores = out[1].detach().cpu().numpy().squeeze()

            # Ensure shape (N,)
            scores = np.asarray(scores)
            if scores.ndim > 1 and scores.shape[-1] == 1:
                scores = scores.squeeze(-1)

            # Convert logits → probabilities if needed
            y_score = scores if logits_are_probs else _sigmoid(scores)

            # Compute metric
            val = metric_fn(y_true, y_score)
            stats.loc[c, t] = val

    return stats 


def plot_metric_heatmap(stats: pd.DataFrame, title: str, cmap="viridis"):

    #MISSING IS WEIGHTED COMBO
    
    fig, ax = plt.subplots(figsize=(8, 6))
    # Mask NaNs to show holes where undefined
    data = stats.to_numpy(dtype=float)
    im = ax.imshow(data, aspect="auto", cmap=cmap, interpolation="nearest")
    ax.set_xticks(np.arange(stats.shape[1]))
    ax.set_yticks(np.arange(stats.shape[0]))
    ax.set_xticklabels([f"{x}" for x in stats.columns])
    ax.set_yticklabels([f"{y}" for y in stats.index])
    ax.set_xlabel("Timepoint")
    ax.set_ylabel("Concentration")
    ax.set_title(title)
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Metric")
    plt.tight_layout()
    plt.show()


def instantiate_model_from_metadata(ModelClass, metadata: dict, input_dim: int) -> nn.Module:
    """
    Map your saved hyperparameters (metadata) to ModelClass kwargs.
    Adjust these keys to match what you logged in artifact.metadata.
    """
    # Example mapping — change keys to what you actually saved:
    kwargs = dict(
        input_dim=input_dim,
        trunk_layers=metadata.get("trunk_layers", 3),
        trunk_dim=metadata.get("trunk_dim", 128),
        reg_layers=metadata.get("reg_layers", 1),
        reg_hidden=metadata.get("reg_hidden", 64),
        cls_layers=metadata.get("cls_layers", 1),
        cls_hidden=metadata.get("cls_hidden", 64),
    )
    return ModelClass(**kwargs)



def get_artifact_and_config(artifact_path: str, *, version: str | None = None):
    """
    Download a W&B artifact and return (local_dir, artifact_metadata).
    artifact_path: e.g. 'entity/project/artifacts/model_artifact_name'
    version: optional version like ':latest' or ':v12' (if not included in path)
    """
    import wandb
    api = wandb.Api()
    full = artifact_path if (version is None or artifact_path.endswith((':latest', ':best')) or artifact_path.rsplit(':',1)[-1].startswith('v')) else f"{artifact_path}{version}"
    art = api.artifact(full)
    local_dir = art.download()
    metadata = dict(art.metadata or {})
    return local_dir, metadata

def infer_model_file(local_dir: str, patterns=(".pt", ".pth", ".bin")) -> str:
    """Find a likely model state file inside the downloaded artifact directory."""
    for ext in patterns:
        files = glob.glob(os.path.join(local_dir, f"*{ext}"))
        if files:
            # pick the largest by size (often the weights)
            return max(files, key=os.path.getsize)
    raise FileNotFoundError(f"No model file with extensions {patterns} found in {local_dir}")


def load_model_from_artifact(
    artifact_path: str,
    *,
    version: str | None = None,
    ModelClass=MultiHeadNet,
    input_dim: int | None = None,
    device: torch.device | str = "cpu",
):
    """
    Download an artifact, read metadata, instantiate ModelClass, and load state_dict.

    Returns
    -------
    model : nn.Module
    metadata : dict
    model_file : str
    """
    local_dir, metadata = get_artifact_and_config(artifact_path, version=version)
    model_file = infer_model_file(local_dir)

    if input_dim is None:
        raise ValueError("You must supply input_dim (number of feature columns passed to the model).")

    model = instantiate_model_from_metadata(ModelClass, metadata, input_dim=input_dim)
    state = torch.load(model_file, map_location="cpu")
    # Accept state dict or full checkpoint
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model, metadata, model_file


# Example

In [ ]:
'''
# A) Compute a heatmap
stats = compute_cellwise_metric_heatmap(
    df=df_test,
    model=model,
    cols_rem=cols_rem,                 # e.g., ['Concentration','Timepoint','is_Active', ...]
    metric_name="avg_precision",       # or 'auc' / 'f1' / 'recall'
    target_col="is_Active",
    device="cuda:0",
    cls_threshold=0.5,
    logits_are_probs=False,            # set True if your head already outputs probabilities
)
plot_metric_heatmap(stats, title="AP by (Conc, Time)")

# B) Plot confusion matrix (contingency table)
plot_contingency_table_from_model(
    model=model,
    df=df_test,
    cols_rem=cols_rem,
    target_col="is_Active",
    device="cuda:0",
    thr=0.5,
    title="Confusion Matrix (Test)",
    logits_are_probs=False,
)

# C) Load the model back from W&B and run inference
# artifact_path looks like: 'your-entity/your-project/model_artifacts/multihead_<runid>'
model, meta, model_file = load_model_from_artifact(
    artifact_path="my-entity/my-project/multihead_artifact_name:v12",
    ModelClass=MultiHeadMLP,   # replace with your actual class if different
    input_dim=X_test.shape[1], # number of input features you pass to model
    device="cuda:0",
)
'''